In [12]:
# -----------------------------------------------------------------------------------------
# CELL 1: CONFIGURATION & METRO MAPPING
# -----------------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import zipfile
import io
import gcsfs
from google.cloud import storage

# --- 1. DATA PATHS ---
icao_metro_path = "gs://agntworks-data-dev/wheelsup/processed/transformed_icao_metro.csv"

target_files = ["gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2026.zip"]

# --- 2. LOAD METRO MAPPING ---
print("Loading ICAO to Metro mapping...")
df_map = pd.read_csv(icao_metro_path)
df_map.columns = [c.lower() for c in df_map.columns]

# Dictionary for fast O(1) lookup
metro_map = dict(zip(df_map['icao'], df_map['metro']))
print(f"✅ Metro mapping loaded: {len(metro_map)} ICAOs ready.")

# --- 3. TIMEZONE & FEATURE LOGIC ---
def process_time_features(df):
    """ Converts Zulu to ET and extracts core time features """
    # Convert string timestamps to ET Datetime objects
    df['FlightDate_ET'] = pd.to_datetime(df['FlightDate_utc'], utc=True).dt.tz_convert('US/Eastern')
    
    # Extract essential features for TOD/DOW analysis
    df['year'] = df['FlightDate_ET'].dt.year
    df['month'] = df['FlightDate_ET'].dt.month
    df['dow'] = df['FlightDate_ET'].dt.day_name()
    df['hour'] = df['FlightDate_ET'].dt.hour
    
    return df

print("🚀 Cell 1 Complete: Metro Mapping and simplified Time Logic established.")

Loading ICAO to Metro mapping...
✅ Metro mapping loaded: 16002 ICAOs ready.
🚀 Cell 1 Complete: Metro Mapping and simplified Time Logic established.


In [13]:
# --- CELL 2: RAW INSPECTOR ---
import zipfile
import io
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem()

# 1. Open the first archive for a schema check
first_zip = target_files[0]
print(f"🔍 Inspecting: {first_zip}")

with fs.open(first_zip) as f:
    with zipfile.ZipFile(io.BytesIO(f.read())) as z:
        # Get the first CSV name in the archive
        all_files = [name for name in z.namelist() if name.endswith('.csv')]
        first_csv = all_files[0]
        
        # 2. Load just the first 5 rows to verify columns
        sample_df = pd.read_csv(z.open(first_csv), nrows=5)

# 3. LIST ALL COLUMNS
print(f"\n📋 ALL AVAILABLE COLUMNS IN {first_csv}:")
print("-" * 50)
for i, col in enumerate(sample_df.columns):
    print(f"{i+1}. {col}")
print("-" * 50)

# Show the raw data sample
display(sample_df)

🔍 Inspecting: gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2026.zip

📋 ALL AVAILABLE COLUMNS IN WINGX_Jan26-Mar26.csv:
--------------------------------------------------
1. FlightDate_utc
2. ArrivalDate_utc
3. Operator
4. FromAirport
5. FromCity
6. FromState
7. ToAirport
8. ToCity
9. ToState
10. Hours
11. aircraft_tailsign
12. aircraft_tailsign_certification
13. operator_type
14. aircraft_icao_code
15. aircraft_type
16. aircraft_model
17. aircraft_segment
18. fuel_uplift_usg
--------------------------------------------------


,FlightDate_utc,ArrivalDate_utc,Operator,FromAirport,FromCity,FromState,ToAirport,ToCity,ToState,Hours,aircraft_tailsign,aircraft_tailsign_certification,operator_type,aircraft_icao_code,aircraft_type,aircraft_model,aircraft_segment,fuel_uplift_usg
0,2026-03-31T07:44:00.000Z,2026-03-31T09:20:00.000Z,ProAir Aviation GmbH,LSGG,Geneva,NaN,LHBP,Budapest,NaN,1.600000,DCLIK,Commercial/ CAT / Part 135,Aircraft Management,C25C,Cessna-Citation CJ4,Citation CJ4 Gen2,Light Jet,307.20
1,2026-03-31T20:45:00.000Z,2026-03-31T22:01:00.000Z,CMH Services Aviation Inc,KORL,Orlando,FL,KTYS,Knoxville,TN,1.266667,N74CH,Part 91 / Non Commercial,Private Flight Department,E550,Embraer-Legacy 500 / Praetor 600,Praetor 600,Super Midsize Jet,393.93
2,2026-03-31T14:27:52.000Z,2026-03-31T15:52:44.000Z,Deer Jet,ZSFZ,Fuzhou,35,ZGSZ,Shenzhen,44,1.414444,B8302,Part 91 / Non Commercial,Aircraft Management,GLF5,Gulfstream-GV/500/550,G550,Ultra Long Range Jet,603.97
3,2026-03-31T22:12:30.000Z,2026-04-01T00:13:38.000Z,Beacon Capital Partners LLC,MMSD,Los Cabos,BCS,KSBD,San Bernardino,CA,2.018889,N119AF,Part 91 / Non Commercial,Corporate Flight Department,GLF4,Gulfstream G300/350/400/450,GIV-SP,Heavy Jet,969.07
4,2026-03-31T13:31:00.000Z,2026-03-31T14:13:00.000Z,NetJets,KDAL,Dallas,TX,KDWH,Houston,TX,0.700000,N271QS,Part 91K / Fractional Ownership,Fractional Ownership,E545,Embraer-Legacy 450 / Praetor 500,Praetor 500,Midsize Jet,210.00


In [16]:
# -----------------------------------------------------------------------------------------
# CELL 3: FINAL REVENUE PIPELINE
# FILTERING: US-Domestic Only (K-Prefix), LJ & SMID Segments, Hours >= 0.5
# -----------------------------------------------------------------------------------------
import zipfile
import io
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem()
all_frames = []

# 1. Define specific columns based on your Cell 2 Inspector
final_cols = [
    'FlightDate_utc', 'Hours', 'Operator', 'FromAirport', 'ToAirport', 
    'FromState', 'ToState', 'aircraft_tailsign', 'aircraft_model', 
    'aircraft_segment', 'fuel_uplift_usg'
]

# 2. Define explicit types to fix DtypeWarnings
dtype_dict = {
    'Operator': str, 'FromAirport': str, 'ToAirport': str, 
    'FromState': str, 'ToState': str, 'aircraft_tailsign': str, 
    'aircraft_model': str, 'aircraft_segment': str, 'Hours': float
}

for zip_path in target_files:
    print(f"\n📂 OPENING ARCHIVE: {zip_path}")
    with fs.open(zip_path) as f:
        with zipfile.ZipFile(io.BytesIO(f.read())) as z:
            csv_files = [name for name in z.namelist() if name.endswith('.csv')]
            for i, member in enumerate(csv_files):
                print(f"   [{i+1}/{len(csv_files)}] 📄 Processing: {member}...", end="\r")
                
                # Load with explicit types
                df = pd.read_csv(z.open(member), usecols=final_cols, dtype=dtype_dict, low_memory=False)
                
                # --- APPLY FILTERS ---
                # 1. US Domestic Only (Both airports must start with 'K')
                # 2. Target Segments (Light Jet & Super Midsize Jet)
                # 3. Revenue Hours (>= 0.5)
                mask = (
                    (df['FromAirport'].str.startswith('K', na=False)) & 
                    (df['ToAirport'].str.startswith('K', na=False)) &
                    (df['aircraft_segment'].isin(['Light Jet', 'Super Midsize Jet'])) & 
                    (df['Hours'] >= 0.5)
                )
                df = df[mask]
                
                if not df.empty:
                    all_frames.append(df)

print(f"\n✅ Finished processing all files.")

# 3. CONSOLIDATE
print("\nMerging US-Domestic segments into master dataframe...")
interim_df = pd.concat(all_frames, ignore_index=True)

# 4. ENRICH (Time Features)
print("Applying ET timezone conversion and time features...")
interim_df = process_time_features(interim_df)

# 5. PERSIST
# Renamed path to reflect US-Domestic focus and Metro terminology
INTERIM_PATH = "Data_26.parquet"
interim_df.to_parquet(INTERIM_PATH, index=False)

print(f"\n✨ PIPELINE COMPLETE")
print(f"---------------------------------")
print(f"Total US Revenue Rows: {len(interim_df):,}")
print(f"Columns Verified: {list(interim_df.columns)}")


📂 OPENING ARCHIVE: gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2026.zip
   [1/1] 📄 Processing: WINGX_Jan26-Mar26.csv...
✅ Finished processing all files.

Merging US-Domestic segments into master dataframe...
Applying ET timezone conversion and time features...

✨ PIPELINE COMPLETE
---------------------------------
Total US Revenue Rows: 274,096
Columns Verified: ['FlightDate_utc', 'Operator', 'FromAirport', 'FromState', 'ToAirport', 'ToState', 'Hours', 'aircraft_tailsign', 'aircraft_model', 'aircraft_segment', 'fuel_uplift_usg', 'FlightDate_ET', 'year', 'month', 'dow', 'hour']


In [17]:
# -----------------------------------------------------------------------------------------
# CELL 4: PARQUET IMPORT & FINAL INTEGRITY CHECK
# FILTERING: US-Domestic Only (K-Prefix), LJ & SMID Segments, Hours >= 0.5
# -----------------------------------------------------------------------------------------
import os
from google.cloud import storage

# 1. Import the cleaned US-Domestic Parquet (Matched to Cell 3 Output)
INTERIM_PATH = "Data_26.parquet"
df_check = pd.read_parquet(INTERIM_PATH)

# 2. Check File Size (On GCS)
# blob = storage.Blob.from_string(INTERIM_PATH, client=storage.Client())
# blob.reload()
# file_size_mb = blob.size / (1024 ** 2)

# 3. Row & Memory Audit
row_count = len(df_check)
mem_usage_mb = df_check.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"📁 FILE INTEGRITY REPORT (US-DOMESTIC)")
print(f"---------------------------------")
print(f"GCS Path:      {INTERIM_PATH}")
print(f"Disk Size:     {file_size_mb:.2f} MB")
print(f"Memory Usage:  {mem_usage_mb:.2f} MB")
print(f"Total US Rows: {row_count:,}")

# 4. Validation: Ensure no International 'Leakage'
non_k_origins = df_check[~df_check['FromAirport'].str.startswith('K', na=False)]['FromAirport'].unique()
if len(non_k_origins) == 0:
    print("✅ SUCCESS: Dataset is strictly US-Domestic (K-only).")
else:
    print(f"⚠️ WARNING: Found {len(non_k_origins)} non-K airports. Filter check required.")

# 5. Basic Distribution Stats
print(f"\n📊 DURATION DISTRIBUTION (REVENUE HOURS)")
display(df_check.groupby('aircraft_segment')['Hours'].describe(percentiles=[.05, .5, .95]).round(2))

# 6. Calendar Coverage Verification
print(f"\n🗓️ CALENDAR CONTINUITY CHECK")
display(df_check.groupby(['year', 'month']).size().unstack(fill_value=0))

ValueError: URI pattern must be gs://bucket/object

In [18]:
# -----------------------------------------------------------------------------------------
# CELL 5: METRO MAPPING & COVERAGE AUDIT
# FILTERING: US-Domestic Only (K-Prefix), LJ & SMID Segments, Hours >= 0.5
# -----------------------------------------------------------------------------------------

# 1. Ensure we are using the US-Domestic data from Cell 3/4
df_check = pd.read_parquet("Data_26.parquet") 

# 2. Perform Mapping using the 'metro_map' from Cell 1
df_check['FromMetro'] = df_check['FromAirport'].map(metro_map).fillna('OTHER_METRO')
df_check['ToMetro'] = df_check['ToAirport'].map(metro_map).fillna('OTHER_METRO')

# 3. Coverage Stats
total_rows = len(df_check)
origin_matches = (df_check['FromMetro'] != 'OTHER_METRO').sum()
dest_matches = (df_check['ToMetro'] != 'OTHER_METRO').sum()

print(f"🎯 METRO COVERAGE ANALYSIS (US-DOMESTIC)")
print(f"---------------------------------")
print(f"Origin Matched:      {origin_matches:,} / {total_rows:,} ({origin_matches / total_rows:.1%})")
print(f"Destination Matched: {dest_matches:,} / {total_rows:,} ({dest_matches / total_rows:.1%})")

# 4. Top 5 Missing Metros (The Gap we discussed)
print("\n🔍 TOP 5 UNMAPPED US ORIGINS:")
display(df_check[df_check['FromMetro'] == 'OTHER_METRO']['FromAirport'].value_counts().head(5))

# 5. Preview the result
display(df_check[['FromAirport', 'FromMetro', 'ToAirport', 'ToMetro']].head(10))

🎯 METRO COVERAGE ANALYSIS (US-DOMESTIC)
---------------------------------
Origin Matched:      270,044 / 274,096 (98.5%)
Destination Matched: 269,991 / 274,096 (98.5%)

🔍 TOP 5 UNMAPPED US ORIGINS:


FromAirport
KFHB    209
KCQF    198
KT82    186
KF45    182
KDNA    150
Name: count, dtype: int64

,FromAirport,FromMetro,ToAirport,ToMetro
0,KORL,North Florida,KTYS,Atlanta
1,KPSF,Boston,KVQQ,North Florida
2,KGSO,Charlotte,KJZI,Charlotte
3,KSBA,LA Basin,KTOA,LA Basin
4,KVNY,LA Basin,KHOU,Houston
5,KPTK,Detroit,KIAD,DMV
6,KTEB,New York City,KMDW,Chicago
7,KGNV,North Florida,KSRQ,South Florida
8,KLAS,LA Basin,KEGE,Denver
9,KMIA,South Florida,KEYW,South Florida


In [19]:
# -----------------------------------------------------------------------------------------
# CELL 6: PERSIST METRO DATA
# FILTERING: US-Domestic Only (K-Prefix), LJ & SMID Segments, Hours >= 0.5
# -----------------------------------------------------------------------------------------

# Define the new path with the _metro suffix
METRO_PATH = "Data_26.parquet"

# Final check: Ensure the columns are named correctly before saving
# We already have FromMetro and ToMetro from Cell 5. 

# Save the dataframe
df_check.to_parquet(METRO_PATH, index=False)

print(f"✅ Success! Metro-mapped dataset saved.")
print(f"📍 Path: {METRO_PATH}")
print(f"📊 Final US Row Count: {len(df_check):,}")

✅ Success! Metro-mapped dataset saved.
📍 Path: Data_26.parquet
📊 Final US Row Count: 274,096


In [1]:
import pandas as pd

# 1. Define the metadata content based on the filters applied in Cells 3 & 4
metadata_dict = {
    "Filter Category": [
        "Geography", 
        "Aircraft Segments", 
        "Minimum Flight Duration", 
        "Mapping Source", 
        "Data Period"
    ],
    "Applied Criteria": [
        "Strictly US-Domestic (Both FromAirport and ToAirport must start with 'K')", 
        "Light Jet, Super Midsize Jet", 
        "Hours >= 0.5 (Revenue Missions)", 
        "transformed_icao_metro.csv (98.3% coverage)", 
        "Full Year 2024 through Full Year 2025"
    ],
    "Action Taken": [
        "Excluded International & Non-K airports", 
        "Excluded Heavy, Midsize, Turboprop, etc.", 
        "Excluded short hops and positioning legs", 
        "Enriched with Metro Area names", 
        "Consolidated from multiple ZIP archives"
    ]
}

# 2. Create DataFrame
df_metadata = pd.DataFrame(metadata_dict)

# 3. Define path (Same location, same name + '_metadata')
METADATA_PATH = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_metadata.csv"

# 4. Save to GCS
df_metadata.to_csv(METADATA_PATH, index=False)

print(f"✅ Metadata CSV successfully created.")
print(f"📍 Path: {METADATA_PATH}")
display(df_metadata)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


✅ Metadata CSV successfully created.
📍 Path: gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_metadata.csv


,Filter Category,Applied Criteria,Action Taken
0,Geography,Strictly US-Domestic (Both FromAirport and ToA...,Excluded International & Non-K airports
1,Aircraft Segments,"Light Jet, Super Midsize Jet","Excluded Heavy, Midsize, Turboprop, etc."
2,Minimum Flight Duration,Hours >= 0.5 (Revenue Missions),Excluded short hops and positioning legs
3,Mapping Source,transformed_icao_metro.csv (98.3% coverage),Enriched with Metro Area names
4,Data Period,Full Year 2024 through Full Year 2025,Consolidated from multiple ZIP archives
